 # Plan recommender

 You have access to behavior data about subscribers who have already switched to the new plans (from the project for the Statistical Data Analysis course). For this classification task, you need to develop a model that will pick the right plan. Since you’ve already performed the data preprocessing step, you can move straight to creating the model.



 Develop a model with the highest possible *accuracy*. In this project, the threshold for accuracy is 0.75. Check the *accuracy* using the test dataset.

In [2]:
# %%
import pandas as pd
pd.set_option('display.max_columns', None)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


 # 1. Loading data

In [5]:
# %%
df = pd.read_csv('https://code.s3.yandex.net/datasets/users_behavior.csv')
df.head()


,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [6]:
# %%
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None


 # 2. Splitting data into sets

In [7]:
# %%
from sklearn.model_selection import train_test_split

train_valid, test = train_test_split(df, test_size=0.2)
train, valid = train_test_split(train_valid, test_size=0.25)

features_train = train.drop(['is_ultra'], axis=1)
target_train = train['is_ultra']
features_valid = valid.drop(['is_ultra'], axis=1)
target_valid = valid['is_ultra']
features_test = test.drop(['is_ultra'], axis=1)
target_test = test['is_ultra']

print(features_train.shape)
print(features_valid.shape)
print(features_test.shape)


(1928, 4)
(643, 4)
(643, 4)


 # 3. Tuning models

In [8]:
# %%
print("Decision Tree")
for depth in range(1, 11):
    model = DecisionTreeClassifier(max_depth=depth, random_state=12345)
    model.fit(features_train, target_train)
    print("max_depth =", depth)
    print("Train:", model.score(features_train, target_train))
    print("Valid:", model.score(features_valid, target_valid))


Decision Tree
max_depth = 1
Train: 0.7598547717842323
Valid: 0.7356143079315708
max_depth = 2
Train: 0.7951244813278008
Valid: 0.7480559875583204
max_depth = 3
Train: 0.8096473029045643
Valid: 0.7620528771384136
max_depth = 4
Train: 0.821058091286307
Valid: 0.7636080870917574
max_depth = 5
Train: 0.8236514522821576
Valid: 0.7589424572317263
max_depth = 6
Train: 0.8345435684647303
Valid: 0.7527216174183515
max_depth = 7
Train: 0.8454356846473029
Valid: 0.7542768273716952
max_depth = 8
Train: 0.8599585062240664
Valid: 0.749611197511664
max_depth = 9
Train: 0.8739626556016598
Valid: 0.7558320373250389
max_depth = 10
Train: 0.8858921161825726
Valid: 0.7527216174183515


In [9]:
# %%
print("Random Forest")
for estim in range(10, 101, 10):
    model = RandomForestClassifier(n_estimators=estim, random_state=12345)
    model.fit(features_train, target_train)
    print("n_estimators =", estim)
    print("Train:", model.score(features_train, target_train))
    print("Valid:", model.score(features_valid, target_valid))


Random Forest
n_estimators = 10
Train: 0.9808091286307054
Valid: 0.7807153965785381
n_estimators = 20
Train: 0.9948132780082988
Valid: 0.7791601866251944
n_estimators = 30
Train: 0.9989626556016598
Valid: 0.7682737169517885
n_estimators = 40
Train: 0.9989626556016598
Valid: 0.7698289269051322
n_estimators = 50
Train: 1.0
Valid: 0.7744945567651633
n_estimators = 60
Train: 1.0
Valid: 0.7729393468118196
n_estimators = 70
Train: 1.0
Valid: 0.7776049766718507
n_estimators = 80
Train: 1.0
Valid: 0.7807153965785381
n_estimators = 90
Train: 1.0
Valid: 0.7744945567651633
n_estimators = 100
Train: 1.0
Valid: 0.7682737169517885


In [10]:
# %%
print("Logistic Regression")
model = LogisticRegression(random_state=12345)
model.fit(features_train, target_train)
print("Train:", model.score(features_train, target_train))
print("Valid:", model.score(features_valid, target_valid))


Logistic Regression
Train: 0.758298755186722
Valid: 0.7216174183514774


 ### Findings



 - Linear regression shows the worst performance but it is not overfitted

 - Decision tree is overfitted but the accuracy is higher

 - Random forrest is also overfitted but the accuracy is slightly higher compared to decision tree

 # 4. Testing model

In [11]:
# %%
features_full_train = train_valid.drop(['is_ultra'], axis=1)
target_full_train = train_valid['is_ultra']


In [12]:
# %%
model = RandomForestClassifier(n_estimators=80, random_state=12345)
model.fit(features_full_train, target_full_train)
model.score(features_test, target_test)


0.8055987558320373

 # 5. Additional task: sanity check

In [13]:
# %%
df['is_ultra'].value_counts() / df.shape[0]


is_ultra
0    0.693528
1    0.306472
Name: count, dtype: float64

 Sanity check score is ~69%, so the logistic regression hasn't learned much.